# Hi-EF Phase 2: frozen affective method matrix

Run the three predeclared validation-only method candidates from Research Specification v0.2 across all five fixed model seeds. The job never loads the test partition.

Attach `ptrnghieu/hi-ef-features-v2` and the saved output of `ptrnghieu/hief-multiseed-validation`. Enable a T4 GPU and Internet. Run the notebook without editing the candidate list, seeds, loss weights, or training configuration.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/phase2_affective_method_matrix')

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)

assert (FEATURES / '01_00059.pt').exists(), 'Attach ptrnghieu/hi-ef-features-v2'
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Locate the exact frozen five-seed baseline summary by content.
import json

valid = []
for path in Path('/kaggle/input').rglob('validation_matrix_summary.json'):
    try:
        summary = json.loads(path.read_text())
    except Exception:
        continue
    if (
        summary.get('test_evaluated') is False
        and summary.get('seeds') == [42, 123, 456, 789, 1024]
        and set(summary.get('models', [])) == {'context', 'full'}
    ):
        valid.append(path)
if len(valid) != 1:
    raise RuntimeError(
        'Attach exactly one hief-multiseed-validation output; '
        f'found {len(valid)} summaries: {valid}'
    )
BASELINE_SUMMARY = valid[0]
print('Baseline summary:', BASELINE_SUMMARY)

In [ ]:
command = [
    'python', str(REPO / 'experiments/run_affective_method_matrix.py'),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--features-dir', str(FEATURES),
    '--baseline-summary', str(BASELINE_SUMMARY),
    '--output-dir', str(OUTPUT),
    '--variants', 'full_aux', 'affective_bottleneck', 'hybrid_residual',
    '--seeds', '42', '123', '456', '789', '1024',
    '--epochs', '50', '--batch-size', '32', '--workers', '2',
    '--learning-rate', '1e-4', '--weight-decay', '1e-5',
    '--patience', '8', '--d-model', '512',
    '--temporal-layers', '2', '--inter-layers', '2',
    '--dropout', '0.1', '--face-pooling', 'masked',
    '--affect-weight', '0.5',
    '--reconstruction-weight', '0.1',
    '--orthogonality-weight', '0.05',
]
subprocess.run(command, check=True)

In [ ]:
import pandas as pd

summary = json.loads((OUTPUT / 'affective_method_summary.json').read_text())
assert summary['test_evaluated'] is False
assert summary['model_seeds'] == [42, 123, 456, 789, 1024]
assert summary['variants'] == ['full_aux', 'affective_bottleneck', 'hybrid_residual']
display(pd.read_csv(OUTPUT / 'affective_method_matrix.csv'))
print(json.dumps({
    'aggregates': summary['aggregates'],
    'provisional_selection': summary['provisional_selection'],
}, indent=2))

Run only via **Save Version → Save & Run All**. Preserve the complete `/kaggle/working/phase2_affective_method_matrix` directory. The displayed selection is provisional until the predeclared hierarchical audit; do not evaluate the test partition.